# pas_fixe

L?utilisateur fournit la fonction objectif et le point initial. Les derivees sont estimees numeriquement. La fonction retourne aussi un message explicite en cas de non-convergence ou de blocage numerique; la derniere cellule donne un exemple.


In [ ]:
import numpy as np


def pas_fixe(f, x0, alpha=0.05, tol=1e-5, max_iter=500):
    """Minimise f par descente a pas fixe et gradient numerique.

    Retourne (x_opt, f_opt, n_iter, converged, message). converged vaut True
    uniquement si la norme du gradient est <= tol; False signale un arret sans
    convergence, qui n'est pas necessairement une erreur Python.
    """
    x = np.asarray(x0, dtype=float)
    if x.ndim != 1 or x.size == 0 or not np.all(np.isfinite(x)):
        raise ValueError("x0 doit etre un vecteur non vide de valeurs finies.")
    if alpha <= 0 or tol <= 0 or max_iter < 0:
        raise ValueError("alpha et tol doivent etre positifs, max_iter positif ou nul.")

    # Les fonctions a croissance rapide peuvent deborder pendant les essais de pas.
    def evaluer_f(point):
        with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
            return float(f(point))

    def produit_scalaire(a, b):
        with np.errstate(over="ignore", invalid="ignore"):
            return float(np.dot(a, b))

    def norme_vecteur(vecteur):
        return float(np.hypot.reduce(vecteur))

    # Estime chaque derivee partielle par differences finies centrees.
    def gradient_numerique(point):
        gradient = np.empty_like(point)
        pas_base = np.cbrt(np.finfo(float).eps)
        for i in range(point.size):
            pas = pas_base * max(1.0, abs(point[i]))
            point_plus = point.copy()
            point_moins = point.copy()
            point_plus[i] += pas
            point_moins[i] -= pas
            valeur_plus = evaluer_f(point_plus)
            valeur_moins = evaluer_f(point_moins)
            if not np.isfinite(valeur_plus) or not np.isfinite(valeur_moins):
                return None
            gradient[i] = (valeur_plus - valeur_moins) / (2 * pas)
        if not np.all(np.isfinite(gradient)):
            return None
        return gradient
    try:
        valeur = evaluer_f(x)
    except (TypeError, ValueError, OverflowError) as erreur:
        message = f"Impossible d'evaluer f(x0): {erreur}. Verifiez la dimension de x0 et la definition de f."
        return x, np.nan, 0, False, message
    if not np.isfinite(valeur):
        return x, valeur, 0, False, "La fonction n'est pas finie au point initial."

    for iteration in range(max_iter):
        gradient = gradient_numerique(x)
        if gradient is None:
            return x, valeur, iteration, False, "Gradient numerique non fini; arret de l'algorithme."
        if norme_vecteur(gradient) <= tol:
            return x, valeur, iteration, True, "Convergence atteinte."

        with np.errstate(over="ignore", invalid="ignore"):
            x_suivant = x - alpha * gradient
        if not np.all(np.isfinite(x_suivant)):
            return x, valeur, iteration, False, "Les iterates sont devenus non finis; arret de l'algorithme."
        valeur_suivante = evaluer_f(x_suivant)
        if not np.isfinite(valeur_suivante):
            return x, valeur, iteration, False, "La fonction est devenue non finie; arret de l'algorithme."
        x, valeur = x_suivant, valeur_suivante

    gradient = gradient_numerique(x)
    if gradient is not None and norme_vecteur(gradient) <= tol:
        return x, valeur, max_iter, True, "Convergence atteinte."
    if gradient is None:
        return x, valeur, max_iter, False, "Gradient numerique non fini apres le nombre maximal d'iterations."
    return x, valeur, max_iter, False, "Nombre maximal d'iterations atteint sans convergence."


In [ ]:
# Fonction de l'exercice utilisee pour illustrer l'appel de l'algorithme.
def f(X):
    x, y, z, w = X
    return 6*x**2 + 6*x*y + 3*y**2 + y**3 - (1/3)*z**3 + (1/4)*z**4 +w

x0 = np.array([-0.6637, 0.1758, 1.5778, 0.5])  # Ajout de la quatrième composante
x_opt, f_opt, n_iter, converged, message = pas_fixe(
    f, x0, alpha=0.05, tol=1e-5, max_iter=500
)

print("Point obtenu :", x_opt)
print("Valeur de la fonction :", f_opt)
print("Nombre d'iterations :", n_iter)
print("Convergence :", converged)
print("Message :", message)
